# Can Kaggle's free T4 run the Isaac Lab gate?

Nobody knows. This notebook finds out, and it costs nothing.

`docs/setup-cloud.md` writes the T4 off as unpromising rather than proven
impossible, which is a polite way of saying it was never tested. The card
itself is not the problem: Turing has RT cores, unlike the P100. The doubts
are session disk against a 30 GB install, and whether Kaggle's image exposes
Vulkan to the driver. Both are unknown rather than known bad.

**Before you run anything: set Accelerator to `GPU T4 x2` in the sidebar,
and turn Internet on.** Then Run All and read the output.

Every stage below prints a verdict. A failure is a result, not a waste of an
hour: it names what would have to change. Copy the whole output when it
finishes, whichever way it goes.

Budget about an hour of your 30 free weekly GPU hours.

## Stage 0: what card did Kaggle give us

Kaggle hands out T4s and P100s depending on availability, and a P100 ends
the experiment immediately. Check before spending the hour.

In [ ]:
import subprocess


def probe(cmd):
    """Run a command, or return None if it is not installed.

    nvidia-smi is absent, not merely unhappy, when the notebook has no
    accelerator attached. That is the single most likely mistake here and it
    deserves a sentence rather than a traceback.
    """
    try:
        return subprocess.run(cmd, capture_output=True, text=True)
    except FileNotFoundError:
        return None


smi = probe(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
             '--format=csv,noheader'])
if smi is None:
    raise SystemExit(
        'nvidia-smi is not installed, which means this session has no GPU.\n'
        'In the sidebar, set Accelerator to "GPU T4 x2", then Run All again.')

gpu = smi.stdout.strip()
print('GPU:', gpu or '(nvidia-smi returned nothing)')

name = gpu.upper()
if 'P100' in name or 'K80' in name:
    raise SystemExit(
        f'{gpu} has no RT cores, so Isaac Sim cannot render on it. This is not\n'
        'a fixable problem: it is the wrong silicon. Factory reset the session\n'
        'and try again later, Kaggle allocates T4s and P100s by availability.')
elif 'T4' in name:
    print('\nT4. Turing, so it has RT cores. This is the case worth testing.')
else:
    print(f'\n{gpu} is not a card this notebook has an opinion about. Continuing.')

## Stage 1: is there enough disk

Isaac Sim is about 30 GB installed. Kaggle caps `/kaggle/working` well below
that, so the install has to go somewhere else on the container filesystem.
This is the first thing that might simply not fit.

In [ ]:
import shutil

for path in ('/', '/tmp', '/kaggle/working', '/opt'):
    try:
        total, used, free = shutil.disk_usage(path)
        print(f'{path:18s} {free / 2**30:6.1f} GB free of {total / 2**30:6.1f} GB')
    except OSError as exc:
        print(f'{path:18s} {exc}')

free_root = shutil.disk_usage('/tmp').free / 2**30
print()
if free_root < 35:
    print(f'VERDICT: {free_root:.1f} GB free under /tmp. Isaac Sim needs about 30 GB\n'
          'installed plus room for the extension cache, so this is tight to\n'
          'impossible. Continuing anyway, because a real failure is worth more\n'
          'than a prediction.')
else:
    print(f'VERDICT: {free_root:.1f} GB free under /tmp. Enough, on paper.')

## Stage 2: is Vulkan there

Isaac Sim renders through Vulkan. A container can ship the loader and still
have no NVIDIA ICD for it to load, which fails at the renderer rather than at
install time. This is the failure this notebook most expects.

In [ ]:
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y vulkan-tools > /dev/null 2>&1
print('--- vulkaninfo --summary ---')
!vulkaninfo --summary 2>&1 | head -40 || echo '(vulkaninfo failed or is absent)'
print()
print('--- ICD files the loader would use ---')
!ls -la /usr/share/vulkan/icd.d/ /etc/vulkan/icd.d/ 2>&1 | head -20

In [ ]:
import glob
import subprocess

icds = glob.glob('/usr/share/vulkan/icd.d/*nvidia*') + \
       glob.glob('/etc/vulkan/icd.d/*nvidia*')

try:
    result = subprocess.run(['vulkaninfo', '--summary'],
                            capture_output=True, text=True)
    works = result.returncode == 0 and 'NVIDIA' in result.stdout
    ran = True
except FileNotFoundError:
    works, ran = False, False

print('NVIDIA ICD files found:', icds or 'none')
print('vulkaninfo ran:', ran)
print('vulkaninfo reports an NVIDIA device:', works)
print()
if works:
    print('VERDICT: Vulkan is exposed. The main doubt about Kaggle is gone.')
elif not ran:
    print('VERDICT: vulkan-tools would not install, so this says nothing either\n'
          'way. Isaac Sim ships its own loader, so continue and let the gate\n'
          'answer the question instead.')
elif icds:
    print('VERDICT: an ICD exists but vulkaninfo could not use it. Often a driver\n'
          'and loader version mismatch. Worth continuing to see how far it gets.')
else:
    print('VERDICT: no NVIDIA Vulkan ICD. Isaac Sim will probably fail at the\n'
          'renderer. Continuing so the failure is observed rather than predicted,\n'
          'but this is very likely the end of the Kaggle route.')

## Stage 3: install Isaac Sim

The slow part, roughly 20 to 30 minutes. Installed to `/tmp` rather than
`/kaggle/working` because of the size cap.

The versions match `scripts/setup_cloud.sh`, so a result here transfers to a
rented box.

In [ ]:
import os

os.environ['PIP_ROOT_USER_ACTION'] = 'ignore'
os.environ['TMPDIR'] = '/tmp/pipbuild'
os.makedirs('/tmp/pipbuild', exist_ok=True)

# The same pins setup_cloud.sh uses, so a result here means something there.
!pip install -q 'torch==2.5.1' --index-url https://download.pytorch.org/whl/cu121
!pip install -q 'isaacsim[all,extscache]==4.5.0' --extra-index-url https://pypi.nvidia.com

In [ ]:
!git clone -q --depth 1 --branch v2.0.2 https://github.com/isaac-sim/IsaacLab.git /tmp/IsaacLab
!cd /tmp/IsaacLab && ./isaaclab.sh --install 2>&1 | tail -20

## Stage 4: install this project

In [ ]:
!git clone -q https://github.com/abyyworld/isaac-grasp-scaling.git /tmp/isaac-grasp-scaling
!pip install -q -e '/tmp/isaac-grasp-scaling[dev]'
!cd /tmp/isaac-grasp-scaling && python scripts/fetch_assets.py

## Stage 5: the gate

Six Isaac stages, none of which has ever been executed anywhere. `--num-envs 4`
keeps it small, since the question is whether it runs at all rather than how
fast.

It stops at the first failure and names it. That name is the output that
matters.

In [ ]:
!cd /tmp/isaac-grasp-scaling && python scripts/check_setup.py --isaac --num-envs 4

## What to do with the result

**Copy the entire output of this notebook**, whichever way it went.

| what happened | what it means |
|---|---|
| all six stages pass | the port works, and it works on free hardware. That is a better result than the rented box was going to give |
| fails at `isaac:import` | the install did not fit or did not complete. Read stages 1 and 3 |
| fails at `isaac:launch` or `isaac:camera` | almost certainly Vulkan. Compare against stage 2 |
| fails at `isaac:object geometry` | **the interesting one.** PhysX is not picking up runtime rescales of collision shapes. The fallback is in `docs/design.md` section 6 and is a change to one module. This failure is worth the same on a T4 as on a 4090 |
| fails at `isaac:execute` | the IK controller never converged. Joint names, body names, or the Jacobian index |

The last three are real findings about the port, and a T4 answers them as well
as a rented 4090 would. Only the first two are Kaggle's fault rather than the
code's.

If this route dies at Vulkan, that is what `docs/setup-cloud.md` should say
instead of the current hedge, and renting becomes the only option.